# 06 — GroupBy / Merge / Pivot Table / 缺失值处理

## 这一讲为什么重要？

前面学了 Pandas 基本操作，但真实的数据分析不是你一个人对着一个干净表格算均值。
真实场景是：

- 你有**多张表**（股票信息表、行情表、财务表），需要**合并**起来分析
- 你需要按**行业/市值分组**统计，而不是整体算一个均值
- 你需要把**行数据转成列**（透视表），方便比较
- 数据一定有**缺失**，你得知道什么时候删、什么时候填

这四个主题是 pandas 数据处理中最常见的操作，**每个量化项目都会用到**。

### 本讲结构

```
1. GroupBy   — 分组聚合（「按行业看平均PE」）
2. Merge     — 表连接（「把股价表和财务表合并」）
3. Pivot     — 透视表（「行转列，看不同行业×年份的收益」）
4. Missing   — 缺失值（「数据有缺口怎么办」）
```

> 💡 每个部分都有「常见坑」表格——这些坑我都踩过，仔细看能帮你省很多时间。


In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.2f}'.format)
### 一句话理解 GroupBy

> GroupBy = **「分堆 → 每堆算一个数 → 把结果拼回来」**

类比：你把一叠扑克牌按花色分成 4 堆，每堆数有多少张 —— 这就是 GroupBy。

在量化里：按行业分组 → 每组算平均 PE → 找出最低估的行业。


---
## 第一部分：GroupBy —— 分组聚合

### 1.1 核心概念

`groupby` 的三步曲：**Split → Apply → Combine**

1. **Split**：按某个键（列）把数据拆成若干组
2. **Apply**：对每组独立执行某个操作（求和、均值、自定义函数...）
3. **Combine**：把各组的结果拼回一张表

```
原始数据                  Split                Apply (求均值)        Combine
部门   姓名   工资          [技术部]             技术部 均值=9500
技术   张三   10000   →     [销售部]        →    销售部 均值=7000    →  部门   平均工资
销售   李四   7000         [行政部]             行政部 均值=7500       技术   9500
技术   王五   9000                                                      销售   7000
行政   赵六   7500                                                      行政   7500
```
### 与 SQL 的类比（如果你学过 SQL）

```sql
SELECT 行业, AVG(PE) FROM stocks GROUP BY 行业;
```

等价于：

```python
df.groupby('行业')['PE'].mean()
```

没学过 SQL 也没关系——下面的例子会让你明白。


In [ ]:
# 模拟一份销售订单数据
np.random.seed(42)
n = 30
df = pd.DataFrame({
    '订单日期': pd.date_range('2025-01-01', periods=n, freq='3D'),
    '省份': np.random.choice(['广东', '浙江', '江苏', '北京'], n),
    '品类': np.random.choice(['家电', '服装', '食品'], n),
    '销售额': np.random.randint(1000, 10000, n),
    '数量': np.random.randint(1, 20, n),
})
df.head(10)

### 1.2 单列分组 + 单聚合

In [ ]:
# 按省份分组，求销售额总和
df.groupby('省份')['销售额'].sum()

In [ ]:
# 按品类分组，求销售额均值
df.groupby('品类')['销售额'].mean()

### 1.3 多列分组（层次化索引）

In [ ]:
# 按 "省份 + 品类" 分组，求销售额总和
df.groupby(['省份', '品类'])['销售额'].sum()

In [ ]:
# 用 unstack() 把层次化索引展开成二维表（就是初级版透视表）
df.groupby(['省份', '品类'])['销售额'].sum().unstack(fill_value=0)

### 1.4 agg() —— 同时对多列应用多个聚合函数

In [ ]:
# 对销售额做 sum 和 mean，对数量做 sum 和 count
df.groupby('品类').agg({
    '销售额': ['sum', 'mean', 'std'],
    '数量': ['sum', 'count']
})

In [ ]:
# 对同一列应用多个函数也可以这样写
df.groupby('省份')['销售额'].agg(['sum', 'mean', 'min', 'max', 'count'])

### 1.5 transform() —— 保持原表行数不变

`transform` 返回和原 DataFrame **行数相同** 的结果，适合做"组内标准化"。

In [ ]:
# 计算每个品类的平均销售额，广播回原表的每一行
df['品类均价'] = df.groupby('品类')['销售额'].transform('mean')

# 计算每个订单偏离品类均值的百分比
df['偏离%'] = (df['销售额'] - df['品类均价']) / df['品类均价'] * 100

df[['省份', '品类', '销售额', '品类均价', '偏离%']].head(10)

### 1.6 filter() —— 按组筛选

筛选出满足条件的**整组数据**（不是行级筛选）。

In [ ]:
# 只保留销售额总和 > 30000 的省份
df.groupby('省份').filter(lambda g: g['销售额'].sum() > 30000)

### 1.7 apply() —— 自定义函数

当内置聚合函数不够用时，用 `apply` 写任意逻辑。

In [ ]:
# 每组取销售额最高的前 2 条记录
def top_n(group, n=2, col='销售额'):
    return group.nlargest(n, col)

df.groupby('品类', group_keys=False).apply(top_n, n=2)[['品类', '省份', '销售额']]

### ⚠️ GroupBy 常见坑

| 坑 | 说明 |
|---|---|
| `groupby` 默认把分组键变成索引 | 用 `as_index=False` 或 `.reset_index()` 恢复为列 |
| 分组后 NaN 被默认丢弃 | `dropna=False` 保留 NaN 组 |
| `transform` vs `apply` 容易混 | `transform` 返回等长 Series；`apply` 返回任意形状 |
| 对分组结果的列取值 | `df.groupby('A')['B']` 是 SeriesGroupBy，`df.groupby('A')[['B','C']]` 是 DataFrameGroupBy，行为不同 |
> 🔑 **最重要的坑：** `groupby` 之后的结果不是 DataFrame！它是一个「GroupBy 对象」，
> 必须再调用 `.mean()` / `.sum()` 等方法才会真正执行计算。这和 SQL 的「惰性求值」是一样的。


In [ ]:
# 坑1：as_index=False 保持列结构
df.groupby('品类', as_index=False)['销售额'].sum()

In [ ]:
# 坑2：含 NaN 的分组——默认 dropna=True 会丢弃 NaN 组
df2 = pd.DataFrame({
    'key': ['A', 'A', 'B', np.nan, 'B'],
    'val': [10, 20, 30, 40, 50]
})
print('dropna=True (默认):')
print(df2.groupby('key')['val'].sum())
print('\ndropna=False:')
print(df2.groupby('key', dropna=False)['val'].sum())

---
## 第二部分：Merge / Join / Concat —— 表拼接

## 2. Merge —— 表连接

### 现实场景

你有一张「股票基本信息表」（代码、名称、行业）和一张「每日行情表」（代码、日期、价格）。
你想看「每个行业每天的涨跌幅」——这就需要把两张表按「代码」连接起来。

### 三种拼接方式的对比

| 方法 | 用途 | 比喻 |
|---|---|---|
| `pd.concat()` | 纵向堆叠或横向拼接 | 把两张表"粘"在一起 |
| `pd.merge()` | 按键（列值）关联 | SQL 的 JOIN |
| `df.join()` | 按索引关联 | merge 的索引版快捷方式 |

**核心概念 — 四种连接方式：**

```
左表         右表         inner (交集)    left             right            outer (并集)
A  B          A  C         A  B  C        A  B  C          A  B  C          A  B  C
1  a          1  x         1  a  x        1  a  x          1  a  x          1  a  x
2  b          3  y                         2  b  NaN        3  NaN y         2  b  NaN
                                                   ← 只保留左表的键 →   3  NaN y
```

In [ ]:
# 构造模拟数据
orders = pd.DataFrame({
    '订单ID': [1, 2, 3, 4, 5],
    '客户ID': ['C01', 'C02', 'C01', 'C03', 'C04'],
    '金额': [500, 300, 800, 200, 950],
    '日期': pd.to_datetime(['2025-01-01', '2025-01-02', '2025-01-03', '2025-01-03', '2025-01-05'])
})

customers = pd.DataFrame({
    '客户ID': ['C01', 'C02', 'C03', 'C05'],
    '姓名': ['张三', '李四', '王五', '赵六'],
    '城市': ['北京', '上海', '广州', '深圳'],
    '等级': ['VIP', '普通', '普通', 'VIP']
})

print('=== 订单表 ===')
display(orders)
print('=== 客户表 ===')
display(customers)

### 2.2 merge() —— 四种连接方式演示
### 用 Venn 图理解四种连接

```
 左表 A      右表 B       inner        left         right        outer
 ┌────┐     ┌────┐       ┌──┐        ┌────┐        ┌────┐       ┌──────┐
 │A   │     │   B│       │AB│        │A   │        │   B│       │A    B│
 │    │  +  │    │   =   └──┘        │    │        │    │       │      │
 └────┘     └────┘      只有交集    保留左表全部  保留右表全部  保留全部
```

- **inner**（默认）：只保留能匹配上的行
- **left**：保留左表所有行，右表没匹配上的填 NaN
- **right**：保留右表所有行，左表没匹配上的填 NaN
- **outer**：保留两表所有行，没匹配上的都填 NaN


In [ ]:
# inner join —— 只保留两表都有的客户 (C01, C02, C03)
# C04 在客户表中不存在，C05 在订单表中不存在 → 都被丢弃
print('=== inner join ===')
display(pd.merge(orders, customers, on='客户ID', how='inner'))

In [ ]:
# left join —— 保留左表（订单表）所有行
# C04 在客户表中不存在 → 客户信息填 NaN
print('=== left join ===')
display(pd.merge(orders, customers, on='客户ID', how='left'))

In [ ]:
# right join —— 保留右表（客户表）所有行
# C05 在订单表中不存在 → 订单信息填 NaN
print('=== right join ===')
display(pd.merge(orders, customers, on='客户ID', how='right'))

In [ ]:
# outer join —— 两表所有行都保留，匹配不上的填 NaN
print('=== outer join ===')
display(pd.merge(orders, customers, on='客户ID', how='outer'))

### 2.3 键名不同的合并

In [ ]:
# 构造名称不一致的情况
left = pd.DataFrame({'user': ['A', 'B', 'C'], 'score': [90, 80, 85]})
right = pd.DataFrame({'uid': ['A', 'B', 'D'], 'age': [25, 30, 22]})

print('左表:')
display(left)
print('右表:')
display(right)

# 使用 left_on + right_on 指定不同的键名
print('合并结果:')
display(pd.merge(left, right, left_on='user', right_on='uid', how='inner'))

### 2.4 concat() —— 纵向/横向拼接

In [ ]:
# 纵向堆叠（axis=0，默认）
jan = pd.DataFrame({'月份': ['1月']*3, '产品': ['A','B','C'], '销量': [100,150,120]})
feb = pd.DataFrame({'月份': ['2月']*3, '产品': ['A','B','C'], '销量': [110,140,130]})

print('=== 纵向拼接 ===')
display(pd.concat([jan, feb], ignore_index=True))

In [ ]:
# 横向拼接（axis=1）
price = pd.DataFrame({'产品': ['A','B','C'], '单价': [10, 20, 30]})
display(pd.concat([jan, price], axis=1))

### 2.5 join() —— 按索引连接

`join()` 是 `merge()` 的快捷方式，当你要按索引而不是按列来连接时使用。

In [ ]:
# 两个 DataFrame 都以产品名为索引
df_a = pd.DataFrame({'收入': [1000, 2000, 1500]}, index=['A', 'B', 'C'])
df_b = pd.DataFrame({'成本': [600, 1200, 900]}, index=['A', 'B', 'C'])

result = df_a.join(df_b)
result['利润'] = result['收入'] - result['成本']
result

### ⚠️ Merge 常见坑

| 坑 | 说明 |
|---|---|
| 键列有重复值时行数会膨胀 | 多对多合并产生笛卡尔积，行数可能暴增 |
| 合并后列名冲突 | 两表有同名非键列时，pandas 自动加 `_x`, `_y` 后缀 |
| 数据类型不一致 | 键列一个 int 一个 str 会匹配不上，先统一类型 |
| `validate` 参数 | 加 `validate='m:1'` 等可提前发现数据质量问题 |

In [ ]:
# 坑：多对多导致行数膨胀
left_dup = pd.DataFrame({'key': ['A', 'A', 'B'], 'val_left': [1, 2, 3]})
right_dup = pd.DataFrame({'key': ['A', 'A', 'B'], 'val_right': [10, 20, 30]})

print(f'左表 {len(left_dup)} 行，右表 {len(right_dup)} 行')
merged = pd.merge(left_dup, right_dup, on='key')
print(f'合并后 {len(merged)} 行（A 产生了 2×2=4 条！）')
display(merged)

---
## 第三部分：Pivot Table —— 数据透视表

## 3. Pivot Table —— 透视表

### 什么场景用透视表？

比如你有一个表：行业 × 年份 × 收益率。你想把「年份」从行变成列，方便横向对比。
这就是透视表：**行变列，交叉位置填聚合值**。

### 3.1 核心概念

透视表本质上是 **"多维度的分组聚合"** 的一种展示形式。它把：
- **行**（index）→ 按什么分组放在行上
- **列**（columns）→ 按什么分组放在列上
- **值**（values）→ 对哪个数值列做聚合
- **聚合函数**（aggfunc）→ 用什么函数聚合（默认 mean）

和 Excel 的透视表逻辑完全一致。

In [ ]:
# 构造一份模拟交易数据
np.random.seed(7)
trade = pd.DataFrame({
    '日期': pd.date_range('2025-01-01', periods=24, freq='W'),
    '地区': np.random.choice(['华东', '华南', '华北', '西南'], 24),
    '品类': np.random.choice(['家电', '服装', '食品', '日化'], 24),
    '渠道': np.random.choice(['线上', '线下'], 24),
    '销售额': np.random.randint(5000, 50000, 24),
    '利润': np.random.randint(500, 5000, 24),
})
trade.head(8)

### 3.2 基础透视：行=地区，列=品类，值=销售额（均值）

In [ ]:
pv1 = pd.pivot_table(
    trade,
    index='地区',       # 行标签
    columns='品类',     # 列标签
    values='销售额',    # 聚合的值
    aggfunc='sum',      # 聚合函数
    fill_value=0        # 无数据的填 0
)
pv1

### 3.3 多值聚合 —— 同时对销售额和利润做透视

In [ ]:
pv2 = pd.pivot_table(
    trade,
    index='地区',
    columns='渠道',
    values=['销售额', '利润'],
    aggfunc='sum',
    fill_value=0
)
pv2

### 3.4 多重聚合 —— 同时看 sum 和 mean

In [ ]:
pv3 = pd.pivot_table(
    trade,
    index='地区',
    columns='品类',
    values='销售额',
    aggfunc=['sum', 'mean'],
    fill_value=0
)
pv3

### 3.5 添加边际汇总 —— margins=True

In [ ]:
pv4 = pd.pivot_table(
    trade,
    index='地区',
    columns='品类',
    values='销售额',
    aggfunc='sum',
    fill_value=0,
    margins=True,           # 添加"总计"行和列
    margins_name='合计'
)
pv4

### 3.6 多月度层次化索引 —— 地区 + 渠道做行

In [ ]:
pv5 = pd.pivot_table(
    trade,
    index=['地区', '渠道'],   # 多层行索引
    columns='品类',
    values='销售额',
    aggfunc='sum',
    fill_value=0
)
pv5

### 3.7 pivot_table vs groupby + unstack

透视表本质上就是 `groupby + unstack` 的语法糖，但透视表多了一些便利功能（margins、多值聚合等）。
以下两行结果等价：

In [ ]:
# 方式1: pivot_table
pv = pd.pivot_table(trade, index='地区', columns='品类', values='销售额', aggfunc='sum', fill_value=0)

# 方式2: groupby + unstack
gb = trade.groupby(['地区', '品类'])['销售额'].sum().unstack(fill_value=0)

print('pivot_table 结果:')
display(pv)
print('groupby + unstack 结果:')
display(gb)

### 3.8 cross-tab (crosstab) —— 频数透视

`pd.crosstab()` 专做**频数统计**，默认聚合函数是 `count`。

In [ ]:
# 统计每个地区 × 每个品类的订单数
ct = pd.crosstab(trade['地区'], trade['品类'], margins=True, margins_name='合计')
ct

---
## 第四部分：缺失值处理策略

缺失值在真实数据中无处不在，处理策略取决于**缺失原因**和**业务场景**。

**核心步骤：**
1. **检测** — 哪里有缺失、缺了多少
2. **理解** — 为什么缺失（MCAR / MAR / MNAR）
3. **处理** — 删除 or 填充（根据业务逻辑选择策略）

In [ ]:
# 构造一份含各种缺失情况的模拟数据
np.random.seed(1)
n = 100
raw = pd.DataFrame({
    '用户ID': range(1, n+1),
    '年龄': np.random.randint(18, 65, n).astype(float),
    '收入': np.random.randint(3000, 50000, n).astype(float),
    '城市': np.random.choice(['北京', '上海', '广州', '深圳', '杭州'], n),
    '会员等级': np.random.choice(['普通', '银卡', '金卡', '钻石'], n),
    '消费金额': np.random.randint(100, 20000, n).astype(float),
    '注册天数': np.random.randint(1, 1000, n),
})

# 人为制造缺失
raw.loc[np.random.choice(n, 12, replace=False), '年龄'] = np.nan      # 12% 年龄缺失
raw.loc[np.random.choice(n, 8, replace=False), '收入'] = np.nan       # 8% 收入缺失
raw.loc[np.random.choice(n, 20, replace=False), '会员等级'] = np.nan  # 20% 会员等级缺失
raw.loc[np.random.choice(n, 5, replace=False), '消费金额'] = np.nan   # 5% 消费金额缺失
raw.loc[np.random.choice(n, 15, replace=False), '注册天数'] = np.nan  # 15% 注册天数缺失

print(f'总行数: {len(raw)}')
raw.head(10)

## 4. 缺失值处理

### 先想清楚一个问题：这些数据为什么缺失？

缺失不是随机的。不同类型的缺失需要不同的处理方式。盲目填充是数据分析中最常见的错误之一。



In [ ]:
# 每列缺失数量
print('=== 每列缺失数量 ===')
print(raw.isnull().sum())

print('\n=== 每列缺失比例 ===')
print((raw.isnull().sum() / len(raw) * 100).round(2))

In [ ]:
# 可视化缺失值分布
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左：缺失量条形图
missing = raw.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
axes[0].barh(missing.index, missing.values, color='steelblue')
axes[0].set_title('各列缺失数量')
axes[0].set_xlabel('缺失行数')

# 右：缺失矩阵热力图（取前 30 行）
sns.heatmap(raw.head(30).isnull(), cbar=False, cmap='Reds', ax=axes[1], yticklabels=False)
axes[1].set_title('缺失热力图（前30行）')

plt.tight_layout()
plt.show()

### 4.2 三种缺失机制

| 类型 | 含义 | 示例 |
|---|---|---|
| **MCAR** (完全随机缺失) | 缺失与任何变量无关 | 传感器偶发故障 |
| **MAR** (随机缺失) | 缺失与其他变量有关，与自身无关 | 年轻人更不愿填收入 |
| **MNAR** (非随机缺失) | 缺失与自身值有关 | 高收入者故意不填 |

> 判断缺失机制需要业务知识，不是代码能自动解决的。先问业务方"为什么缺"，再定策略。

### 4.3 策略一：删除缺失值

In [ ]:
# dropna() 删除任何含缺失值的行
print(f'原始行数: {len(raw)}')
print(f'删除含任何 NaN 的行后: {len(raw.dropna())}')
print(f'删除所有列都是 NaN 的行: {len(raw.dropna(how="all"))}')

# thresh: 至少要有 N 个非 NaN 值才保留
print(f'至少要有 5 个非 NaN 列才保留: {len(raw.dropna(thresh=5))}')

# 指定子集
print(f'只关心年龄列: {len(raw.dropna(subset=["年龄"]))}')

# 删除缺失超过 80% 的列
threshold = 0.8 * len(raw)
cols_to_keep = raw.columns[raw.notnull().sum() >= threshold]
print(f'\n保留的列（缺失<80%）: {list(cols_to_keep)}')

### 4.4 策略二：填充缺失值 (fillna)

这是最常见的处理方式。不同列用不同策略。

In [ ]:
# 先做一份副本，保持原始数据不变
df_fill = raw.copy()

# ---- 数值列填充 ----

# 年龄：用中位数填充（对异常值稳健）
df_fill['年龄'] = df_fill['年龄'].fillna(df_fill['年龄'].median())

# 收入：用均值填充
df_fill['收入'] = df_fill['收入'].fillna(df_fill['收入'].mean())

# 消费金额：用 0 填充（表示"未消费"）
df_fill['消费金额'] = df_fill['消费金额'].fillna(0)

# 注册天数：用特定值填充
df_fill['注册天数'] = df_fill['注册天数'].fillna(-1)

# ---- 分类列填充 ----

# 会员等级：用众数填充
mode_val = df_fill['会员等级'].mode()[0]
df_fill['会员等级'] = df_fill['会员等级'].fillna(mode_val)

# 确认填充结果
print('=== 填充后缺失值数量 ===')
print(df_fill.isnull().sum())

### 4.5 高级填充策略

In [ ]:
# ---- 前向 / 后向填充（常用于时间序列）----
ts = pd.Series([1, np.nan, np.nan, 4, np.nan, 6, np.nan, np.nan, np.nan, 10])
print('原始序列:', ts.values)
print('前向填充:', ts.ffill().values)
print('后向填充:', ts.bfill().values)
print('线性插值:', ts.interpolate().values)

In [ ]:
# ---- 分组建模填充（更精确）----
# 按城市分组，用各城市的收入中位数填充
raw_copy2 = raw.copy()
raw_copy2['收入'] = raw_copy2.groupby('城市')['收入'].transform(
    lambda x: x.fillna(x.median())
)
print('分组后填充，各城市收入缺失量:')
print(raw_copy2.groupby('城市')['收入'].apply(lambda x: x.isnull().sum()))

### 4.6 缺失值填充策略速查表

| 场景 | 推荐策略 | 原因 |
|---|---|---|
| 缺失比例 > 80% | 删除该列 | 信息量太少，填充会引入噪声 |
| 数值列，有异常值 | 中位数填充 | 中位数对极值不敏感 |
| 数值列，近正态分布 | 均值填充 | 均值是好的集中趋势估计 |
| 分类列 | 众数 / 新类别 'Unknown' | 保持分类语义 |
| 时间序列 | ffill / 插值 | 利用时间顺序信息 |
| 金额类（可能为 0） | 0 填充 | "缺失"可能意味着"没发生" |
| 有分组信息 | 分组中位数/众数 | 利用组内同质性提高精度 |
| 建模场景 | KNN / MICE 插补 | 利用变量间关系 |
| 标记缺失类型 | NaN + 添加 is_xxx_missing 列 | 保留"缺失"本身的信息 |

In [ ]:
# 示例：添加"缺失标记列"——保留缺失本身的信息
marked = raw.copy()
marked['年龄_缺失'] = marked['年龄'].isnull().astype(int)
marked['收入_缺失'] = marked['收入'].isnull().astype(int)

# 然后再填充
marked['年龄'] = marked['年龄'].fillna(marked['年龄'].median())
marked['收入'] = marked['收入'].fillna(marked['收入'].median())

marked[['年龄', '年龄_缺失', '收入', '收入_缺失']].head(10)

---
## 综合练习

把四个知识点串起来做一个小案例：
1. 加载两份数据（订单 + 客户）
2. **merge** 连接
3. 检查并处理**缺失值**
4. 用 **groupby** 做客户级别的汇总
5. 用 **pivot_table** 做地区 × 品类分析

In [ ]:
# Step 1 & 2: 构造数据 + merge
np.random.seed(99)

sales = pd.DataFrame({
    'order_id': range(1, 51),
    'customer': np.random.choice(['C001','C002','C003','C004','C005','C006'], 50),
    'product': np.random.choice(['笔记本电脑','手机','平板','耳机'], 50),
    'region': np.random.choice(['华东','华南','华北','西南'], 50),
    'amount': np.random.randint(100, 20000, 50).astype(float),
    'quantity': np.random.randint(1, 5, 50),
})

cust = pd.DataFrame({
    'customer': ['C001','C002','C003','C004','C005','C007'],  # C006 不在客户表，C007 无订单
    'name': ['张三','李四','王五','赵六','孙七','周八'],
    'level': ['VIP','普通','普通','VIP','金卡','普通'],
    'city': ['上海','北京','广州','深圳','杭州',np.nan],  # 故意缺一个
})

# left merge: 保留所有订单
merged = pd.merge(sales, cust, on='customer', how='left')
print(f'合并后形状: {merged.shape}')
print(f'\n缺失值情况:')
print(merged.isnull().sum())
merged.head(8)

In [ ]:
# Step 3: 处理缺失值
print('C006 的客户信息全 NaN —— 因为它不在客户表中')
print('C006 的订单:', merged[merged['customer'] == 'C006']['order_id'].tolist())

# 策略: name 填 '未知客户', level 填 '未知', city 填 '未知'
merged['name'] = merged['name'].fillna('未知客户')
merged['level'] = merged['level'].fillna('未知')
merged['city'] = merged['city'].fillna('未知')

print('\n填充后缺失值:')
print(merged.isnull().sum())

In [ ]:
# Step 4: GroupBy —— 按客户汇总
customer_summary = merged.groupby(['customer', 'name', 'level', 'city']).agg(
    订单数=('order_id', 'count'),
    总金额=('amount', 'sum'),
    平均单价=('amount', 'mean'),
    购买品类数=('product', 'nunique')
).reset_index()

customer_summary.sort_values('总金额', ascending=False)

In [ ]:
# Step 5: Pivot Table —— 地区 × 品类 销售额
region_product = pd.pivot_table(
    merged,
    index='region',
    columns='product',
    values='amount',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='合计'
)
region_product

---
## 总结

| 操作 | 一句话 | SQL 类比 |
|---|---|---|
| `groupby().agg()` | 分组后做聚合 | `GROUP BY` + 聚合函数 |
| `groupby().transform()` | 分组后广播回原表 | 窗口函数（PARTITION BY） |
| `groupby().filter()` | 筛选整组 | `HAVING` |
| `pd.merge(how='left')` | 按列连接 | `LEFT JOIN` |
| `pd.concat()` | 纵向/横向堆叠 | `UNION ALL` / 横向拼接 |
| `pd.pivot_table()` | 多维聚合交叉表 | Excel 透视表 |
| `fillna()` | 缺失值填充 | `COALESCE` / `IFNULL` |
| `dropna()` | 删除缺失行 | `WHERE col IS NOT NULL` |

**记住一个核心原则：** 缺失值的处理策略取决于业务逻辑，不是技术问题。先理解数据为什么缺，再选方法。